In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "desafio_sicredi"

BASE = f"{CATALOG}.{SCHEMA}"

print("Ambiente configurado:", BASE)

Ambiente configurado: workspace.desafio_sicredi


In [0]:
##ler_tabelas_prata

df_clientes = spark.table(
    f"{BASE}.silver_clientes"
)

df_contas = spark.table(
    f"{BASE}.silver_contas"
)

df_cartoes = spark.table(
    f"{BASE}.silver_cartoes"
)

df_transacoes = spark.table(
    f"{BASE}.silver_transacoes_cartao"
)

print("Clientes:", df_clientes.count())
print("Contas:", df_contas.count())
print("Cartões:", df_cartoes.count())
print("Transações:", df_transacoes.count())

Clientes: 2
Contas: 2
Cartões: 3
Transações: 5


In [0]:

%sql

CREATE OR REPLACE TABLE
workspace.desafio_sicredi.gold_transacoes_cartao
USING DELTA
AS

WITH base_transacoes AS (

    SELECT
        id_transacao,
        id_cartao,
        data_transacao,
        valor,
        valor_liquido,
        estornada,
        nome_estabelecimento,
        mcc,
        cidade_estabelecimento,
        data_referencia
    FROM workspace.desafio_sicredi.silver_transacoes_cartao

),

base_cartoes AS (

    SELECT
        id_cartao,
        id_conta,
        bandeira,
        tipo_cartao,
        status_cartao
    FROM workspace.desafio_sicredi.silver_cartoes

),

base_contas AS (

    SELECT
        id_conta,
        id_cliente,
        tipo_conta,
        status_conta
    FROM workspace.desafio_sicredi.silver_contas

),

base_clientes AS (

    SELECT
        id_cliente,
        nome AS nome_cliente,
        cidade AS cidade_cliente,
        estado AS estado_cliente
    FROM workspace.desafio_sicredi.silver_clientes

)

SELECT
    t.id_transacao,
    cli.id_cliente,
    cli.nome_cliente,
    cli.cidade_cliente,
    cli.estado_cliente,

    c.id_conta,
    c.tipo_conta,
    c.status_conta,

    ca.id_cartao,
    ca.bandeira,
    ca.tipo_cartao,
    ca.status_cartao,

    t.data_transacao,
    YEAR(t.data_transacao) AS ano,
    MONTH(t.data_transacao) AS mes,

    t.valor,
    t.valor_liquido,
    t.estornada,

    t.nome_estabelecimento,
    t.mcc,
    t.cidade_estabelecimento,
    t.data_referencia

FROM base_transacoes t

INNER JOIN base_cartoes ca
    ON t.id_cartao = ca.id_cartao

INNER JOIN base_contas c
    ON ca.id_conta = c.id_conta

INNER JOIN base_clientes cli
    ON c.id_cliente = cli.id_cliente;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    id_transacao,
    nome_cliente,
    nome_estabelecimento,
    valor,
    estornada,
    valor_liquido

FROM workspace.desafio_sicredi.gold_transacoes_cartao

ORDER BY id_transacao;

id_transacao,nome_cliente,nome_estabelecimento,valor,estornada,valor_liquido
5001,Ana Maria Silva,SUPERMERCADO RECIFE,150.00,false,150.00
5002,Ana Maria Silva,FARMACIA CENTRAL,80.00,true,0.00
5004,Mariana Costa,POSTO CENTRAL,300.00,false,300.00
5006,Ana Maria Silva,SUPERMERCADO RECIFE,200.00,false,200.00
5008,Ana Maria Silva,LOJA ATRASADA,450.00,false,450.00


In [0]:
%sql

SELECT
    COUNT(*) AS total_transacoes,
    SUM(valor) AS valor_bruto,
    SUM(valor_liquido) AS valor_liquido,
    SUM(
        CASE WHEN estornada THEN 1 ELSE 0 END
    ) AS total_estornadas

FROM workspace.desafio_sicredi.gold_transacoes_cartao;

total_transacoes,valor_bruto,valor_liquido,total_estornadas
5,1180.00,1100.00,1
